In [19]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_nvidia_ai_endpoints import ChatNVIDIA
from langchain_ollama import ChatOllama
from langchain_groq import ChatGroq
from dotenv import load_dotenv
import json
import os

load_dotenv()

True

In [20]:
llm = ChatGoogleGenerativeAI(
    model='gemini-2.0-flash',
    temperature=0
)
llm = ChatGroq(
    model="llama-3.1-8b-instant",
)

# llm = ChatNVIDIA(
#     model="deepseek-ai/deepseek-v4-flash",
#     temperature=1,
#     top_p=0.95,
# )

# llm = ChatOllama(
#     model='gemma4:e2b',
#     temperature=0,
# )

res = llm.invoke("Which is the biggest whale in the world?")
print(res)

content='The biggest whale in the world is the blue whale (Balaenoptera musculus). On average, an adult blue whale can grow up to 82 feet (25 meters) in length and weigh around 150-170 tons. However, the largest blue whale ever recorded was 108 feet (33 meters) long and weighed around 210 tons.' additional_kwargs={} response_metadata={'token_usage': {'completion_tokens': 72, 'prompt_tokens': 44, 'total_tokens': 116, 'completion_time': 0.074009169, 'completion_tokens_details': None, 'prompt_time': 0.003584926, 'prompt_tokens_details': None, 'queue_time': 0.056141913, 'total_time': 0.077594095}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--019f8999-f121-7683-aa3b-badcb10dee82-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 44, 'output_tokens': 72, 'total_tokens': 116}


In [21]:
from langchain_community.document_loaders import (TextLoader, PyPDFLoader)
from langchain_text_splitters import RecursiveCharacterTextSplitter
# loader = TextLoader("./docs/memory.md")
# soul = loader.load()[0].page_content
# print(soul)

loader = PyPDFLoader("./docs/harry-st.pdf")
harry = loader.load()


chars = []
for doc in harry:
    if doc.page_content: chars.append(len(doc.page_content))

avg_chars =  sum(chars) // len(chars)
chunk_size = int(avg_chars * 0.25)
chunk_overlap = int(chunk_size * 0.1)

print(f"{avg_chars}, {chunk_size}, {chunk_overlap}")


1756, 439, 43


In [22]:
import uuid

rts = RecursiveCharacterTextSplitter(separators=[ " "], chunk_size=chunk_size, chunk_overlap=chunk_overlap)
splitted_harry = rts.split_documents(harry)
doc_meta = []
for doc in splitted_harry:
    id = uuid.uuid4()
    doc_meta.append(id)
    doc.metadata['id'] = id

In [23]:
import chromadb
cclient = chromadb.Client()

try: cclient.delete_collection(name="Harry_Potter_and_the_Sorcerers_Stone")
except: print("Collection doesn't exist")
collection = cclient.create_collection(name="Harry_Potter_and_the_Sorcerers_Stone")


In [24]:
collection.add(
    ids=[str(uuid.uuid4()) for _ in splitted_harry],
    documents=[x.page_content for x in splitted_harry]
)

In [25]:
from pprint import pprint
query = "Who is the cat in the first scene, when potter is given to his uncle's family?"
res = collection.query(query_texts=[query], n_results=5)

In [26]:
out = llm.invoke(f"""
    You are acting as an helpful agent which gets the query and from the given information ONLY, returns the summarized answer.
    DO NOT GIVE THE ANSWER OUT OF THE GIVEN DOCUMENTS. IF THE DOCUMENT IS NOT SUFFICENT TO ANSWER THE QUERY, RETURN "The given information is insufficient to answer your query"
    Query: {query}
    Related Documents: {res}
    Give the answer in the following format:
    {{
        content: 'He was doing some work for this person[0]. On behalf of that person who was the headmaster[1] of that place[2].',
        references: [
            <id[0]>,
            <id[1]>,
            <id[2]>,
        ]
    }}
    In above format we see the given output with the reference markers in square brackets which align with the indexes of the references that map to the IDs the information which the fact was soruced from.
    If there is insufficient information, then the references array should be empty ([])
    MAKE SURE THE REFERENCE MARKERS ARE PLACED BESIDE THE RELEVANT INFORMATION WHICH IS SOURCED
    FOLLOW THE ABOVE FORMAT STRICTLY WITHOUT ANY CHANGES. 
    > DO NOT ANSWER IN ANY SORT OF MARKDOWN, JUST PURE JSON STRING.
    > IF THE GIVEN TEXT IS OUT OF THE CONTEXT LENGTH, THEN RETURN "Out of Context Length, Please shorten the message".
""")
try: 
    out = json.loads(out.content)
    pprint(out)
except:
    print("The output wasn't in JSON format")
    print(out.content)

{'content': "The cat in the first scene is the tabby cat he'd spotted that "
            'morning [0]. It was now sitting on his garden wall, and he was '
            'sure it was the same one [0].',
 'references': ['6ad88b03-743f-4822-8858-feee39f8027b',
                '6ad88b03-743f-4822-8858-feee39f8027b']}


In [27]:
for doc, id in zip(res['documents'][0], res['ids'][0]):
    print(f'ID: {id}')
    print(f'Doc: {doc}')
    print()

ID: 6ad88b03-743f-4822-8858-feee39f8027b
Doc: Harry was looking at his family,
for the first time in his life.

ID: ef8d7a1a-490e-4a2e-8bd1-2cd2fd193128
Doc: 4
he was imagining things, which he had never hoped before, because he
didn't approve of imagination.
As he pulled into the driveway of number four, the first thing he saw --
and it didn't improve his mood -- was the tabby cat he'd spotted that
morning. It was now sitting on his garden wall. He was sure it was the
same one; it had the same markings around its eyes.
"Shoo!" said Mr. Dursley loudly. The cat didn't move. It just gave him

ID: 91ac1c07-e284-4043-abf9-cb1578494ff4
Doc: round the door. Ron and Harry stood quite still, both thinking the
same thing -- did the cloak work on cats? After what seemed an age, she
turned and left.
"This isn't safe -- she might have gone for Filch, I bet she heard us.
Come on."
And Ron pulled Harry out of the room.
The snow still hadn't melted the next morning.
"Want to play chess, Harry?" said 

In [28]:
cclient.list_collections()

[Collection(name=Harry_Potter_and_the_Sorcerers_Stone)]